# Phase 08 — Teaching Modes

**Main question:** How do we shape what the VLM does with multimodal evidence?

In Phase 07c we built the full pipeline: retrieve text + visual → VLM → grounded answer.
Teaching modes keep the same retrieval pipeline but replace the default prompt template
with a mode-specific instructional template.

**Modes:**
1. **Explain** — pedagogical explanation at a specified level
2. **Socratic tutor** — guiding questions, no direct answer
3. **Quiz** — generate questions from text + visual evidence
4. **Compare** — compare figures, methods, or claims
5. **Visual evidence** — explain what each figure shows and why it matters

**Package:** `mrta-rag[retrieval,multimodal]`

> **Note:** Sections 2–6 require Ollama running with a vision model.
> Section 1 is VLM-free.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

from mrta import (
    Embedder, VectorStore, chunk_pdf, load_pdf,
    CLIPEmbedder, VisualVectorStore, extract_figures,
    MultimodalRetriever, MultimodalRAG, VLMClient,
)

SAMPLE_PDF = Path("../../tests/fixtures/sample.pdf")
assert SAMPLE_PDF.exists(), f"Sample PDF not found at {SAMPLE_PDF}"
print(f"Sample PDF: {SAMPLE_PDF.resolve()}")

---
## 08.1 Setup: Build the Shared Retrieval Pipeline

All five teaching modes reuse the same retriever.
The mode changes only which Jinja2 template is rendered.

In [ ]:
doc = load_pdf(SAMPLE_PDF)
chunks = chunk_pdf(doc)
embedder = Embedder()
text_store = VectorStore(embedder)
text_store.add(chunks)
print(f"Text index: {text_store.size} chunks")

clip = CLIPEmbedder()
figures = extract_figures(doc)
visual_records = [f.to_evidence_record() for f in figures]
visual_store = VisualVectorStore(clip)
if visual_records:
    visual_store.add(visual_records)
    print(f"Visual index: {visual_store.size} figures")
else:
    print("No raster figures — visual index empty.")

retriever = MultimodalRetriever(
    vector_store=text_store,
    visual_store=visual_store if visual_store.size > 0 else None,
    rrf_k=60,
)

vlm_available = VLMClient.is_available()
print(f"VLM available: {vlm_available}")

QUERY = "What is the role of the attention mechanism?"

---
## 08.2 Explain Mode

**Template:** `teaching_explain.j2`

The VLM is instructed to:
- Teach at undergraduate level
- Define technical terms when they first appear
- Describe figure elements before connecting them to the concept
- Cite evidence as [T#] and [V#]

In [ ]:
if vlm_available:
    vlm = VLMClient()
    mmrag = MultimodalRAG(
        retriever=retriever, vlm=vlm,
        text_top_k=5, visual_top_k=5, fusion_top_k=8,
        teaching_mode="explain",
    )
    result = mmrag.ask(QUERY)

    print(f"Mode: {result.retrieval_mode}  |  Latency: {result.latency_s:.2f}s")
    print(f"Text citations : {[c.label for c in result.text_citations]}")
    print(f"Visual citations: {[c.label for c in result.visual_citations]}")
    print()
    print("--- EXPLANATION ---")
    print(result.answer)
else:
    print("VLM not available. Install: ollama pull qwen2.5vl:latest && ollama serve")

---
## 08.3 Socratic Mode

**Template:** `teaching_socratic.j2`

The VLM poses 3-5 guiding questions grounded in specific evidence.
Each question cites [T#] or [V#]. The VLM must NOT state the answer.

In [ ]:
if vlm_available:
    mmrag_s = MultimodalRAG(
        retriever=retriever, vlm=vlm, teaching_mode="socratic"
    )
    result_s = mmrag_s.ask(QUERY)
    print("--- GUIDING QUESTIONS ---")
    print(result_s.answer)
else:
    print("VLM not available.")

---
## 08.4 Quiz Mode

**Template:** `teaching_quiz.j2`

The template requests 5 questions:
- 2 multiple-choice (4 options, one correct)
- 2 short-answer
- 1 visual-reasoning (if visual evidence is available)

Each question is labelled with its evidence source. Output ends with an ANSWER KEY.

In [ ]:
QUIZ_QUERY = "Explain the architecture of the Transformer model and its components."

if vlm_available:
    mmrag_q = MultimodalRAG(
        retriever=retriever, vlm=vlm, teaching_mode="quiz"
    )
    result_q = mmrag_q.ask(QUIZ_QUERY)
    print("--- QUIZ ---")
    print(result_q.answer)
else:
    print("VLM not available.")

---
## 08.5 Compare Mode

**Template:** `teaching_compare.j2`

For visual comparisons, the VLM receives multiple PIL images.
It describes each figure's elements before comparing them structurally.

In [ ]:
COMPARE_QUERY = "Compare the encoder and decoder architectures shown in the figures."

if vlm_available:
    mmrag_c = MultimodalRAG(
        retriever=retriever, vlm=vlm,
        text_top_k=3, visual_top_k=5, fusion_top_k=8,
        teaching_mode="compare",
    )
    result_c = mmrag_c.ask(COMPARE_QUERY)
    print(f"Visual evidence: {[c.label for c in result_c.visual_citations]}")
    print()
    print("--- COMPARISON ---")
    print(result_c.answer)
else:
    print("VLM not available.")

---
## 08.6 Visual Evidence Mode

**Template:** `teaching_visual_evidence.j2`

For each retrieved figure the VLM:
1. Describes visual elements (axes, curves, labels, annotations)
2. Explains how the figure supports the question
3. Notes limitations or alternative interpretations

In [ ]:
VIS_QUERY = "What architecture is shown in the figures?"

if vlm_available:
    mmrag_v = MultimodalRAG(
        retriever=retriever, vlm=vlm,
        text_top_k=3, visual_top_k=5, fusion_top_k=8,
        teaching_mode="visual_evidence",
    )
    result_v = mmrag_v.ask(VIS_QUERY)
    print(f"Visual citations: {[c.label for c in result_v.visual_citations]}")
    print()
    print("--- VISUAL EVIDENCE ANALYSIS ---")
    print(result_v.answer)
else:
    print("VLM not available.")

---
## 08.7 API Integration

Teaching modes are exposed via `POST /ask` with two new fields:
- `retrieval_mode`: `"text"` (default) or `"multimodal"`
- `teaching_mode`: `"explain"` | `"socratic"` | `"quiz"` | `"compare"` | `"visual_evidence"` | `null`

The response gains `retrieval_mode` (str) and `visual_sources` (list).
Existing clients that send no new fields continue to work unchanged.

In [ ]:
import httpx

payload = {
    "question": "What is the role of the attention mechanism?",
    "retrieval_mode": "multimodal",
    "teaching_mode": "explain",
    "top_k": 5,
}

try:
    r = httpx.post("http://localhost:8000/ask", json=payload, timeout=120)
    r.raise_for_status()
    resp = r.json()
    print(f"retrieval_mode : {resp['retrieval_mode']}")
    print(f"visual_sources : {resp['visual_sources']}")
    print()
    print(resp["answer"][:400])
except Exception as e:
    print(f"API not running: {e}")
    print("Start with: uvicorn apps.api.main:app --reload --port 8000")

---
## Summary

| Mode | Template | VLM instruction |
|---|---|---|
| (none) | `multimodal_rag.j2` | Answer with [T#]/[V#] citations |
| explain | `teaching_explain.j2` | Explain at undergraduate level |
| socratic | `teaching_socratic.j2` | Pose guiding questions, no direct answer |
| quiz | `teaching_quiz.j2` | Generate 5 questions + answer key |
| compare | `teaching_compare.j2` | Compare figures/claims structurally |
| visual_evidence | `teaching_visual_evidence.j2` | Analyse each figure's relevance |

**Key insight:** All modes share the same retrieval pipeline.
Switching modes costs nothing computationally — only the rendered prompt changes.

> Phase 09 evaluates these modes quantitatively.